In [36]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ieee-fraud-detection')

print("Path to competition files:", path)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Path to competition files: /kaggle/input/competitions/ieee-fraud-detection


# 1. Import Framework

In [37]:
#load libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# 2. Load Data

In [ ]:
#Load the Data
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity     = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


# 3. Merge Data

In [ ]:
#Merge the Data
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")

# 4. Memory Optimizatio

In [ ]:
#Memory Optimization
def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
        else:
            df[col] = df[col].astype("category")
    return df

X = reduce_mem(X)
X_test = reduce_mem(X_test)


# 5. Split Data into Features and Target

In [ ]:
#Separate Target and Features
y = train["isFraud"]
X = train.drop("isFraud", axis=1)
X_test = test.copy()


# 6. Align Train And Test Columns

In [ ]:
#Align columns between train and test
missing_cols = [col for col in X.columns if col not in X_test.columns]
X_test = pd.concat([X_test, pd.DataFrame(np.nan, index=X_test.index, columns=missing_cols)], axis=1)
X_test = X_test[X.columns]


# 7. Automatic Column Detection

In [ ]:
# Identify numeric and categorical columns:
numeric_cols = X_train.select_dtypes(include=["int", "float", "float32", "float64"]).columns
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns


# 8. Feature Engineering 

In [ ]:
# --- Time Features ---
X["day"] = X["TransactionDT"] // (24*60*60)
X_test["day"] = X_test["TransactionDT"] // (24*60*60)

X["hour"] = (X["TransactionDT"] // 3600) % 24
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24

# --- Amount Features ---
X["log_amt"] = np.log1p(X["TransactionAmt"])
X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])

X["amt_per_day"] = X["TransactionAmt"] / (X["day"] + 1)
X_test["amt_per_day"] = X_test["TransactionAmt"] / (X_test["day"] + 1)

# --- Frequency Encoding ---
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]

for col in freq_cols:
    freq = X[col].value_counts()
    X[col + "_freq"] = X[col].map(freq)
    X_test[col + "_freq"] = X_test[col].map(freq)

# --- Device Features ---
X["device_name"] = X["DeviceInfo"].str.split("/", expand=True)[0]
X_test["device_name"] = X_test["DeviceInfo"].str.split("/", expand=True)[0]

X["device_version"] = X["DeviceInfo"].str.split("/", expand=True)[1]
X_test["device_version"] = X_test["DeviceInfo"].str.split("/", expand=True)[1]

# --- Email Domain Grouping ---
email_map = {
    'gmail.com': 'google', 'gmail': 'google',
    'yahoo.com': 'yahoo', 'yahoo.com.mx': 'yahoo',
    'hotmail.com': 'microsoft', 'outlook.com': 'microsoft'
}

X["P_emaildomain_group"] = X["P_emaildomain"].map(email_map)
X_test["P_emaildomain_group"] = X_test["P_emaildomain"].map(email_map)


In [ ]:
#Feature Engineering time features
X["day"] = X["TransactionDT"] // (24*60*60)
X["hour"] = (X["TransactionDT"] // 3600) % 24

X_test["day"] = X_test["TransactionDT"] // (24*60*60)
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24


# 9. Encode categorical features

In [ ]:

categorical_cols = X.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))



# 10. Convert Numeric Columns To Float32

In [ ]:
X[numeric_cols] = X[numeric_cols].astype("float32")
X_test[numeric_cols] = X_test[numeric_cols].astype("float32")


# 11. Handle missing values

In [ ]:
#Handle Missing Values
num_cols = X.select_dtypes(include=["int", "float"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

X[num_cols] = X[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

X[cat_cols] = X[cat_cols].fillna("missing")
X_test[cat_cols] = X_test[cat_cols].fillna("missing")


# 12.Fill Missing Values

In [ ]:
X = X.fillna(-1)
X_test = X_test.fillna(-1)


# 13. Train And Validation Split

In [ ]:
#train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# 14. Lightgbm Datasets

In [ ]:
train_set = lgb.Dataset(X_train, y_train)
valid_set = lgb.Dataset(X_valid, y_valid)


# 15 Lightgbm Parameters

In [ ]:
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 256,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1
}


# 16 Train The Model

In [ ]:
model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=20000,
    early_stopping_rounds=500
)



# 17. Validation Predictions and AUC

In [ ]:
valid_pred = model.predict(X_valid)
auc = roc_auc_score(y_valid, valid_pred)
print("Validation AUC:", auc)


# 18. Test Predictions

In [ ]:
test_pred = model.predict(X_test)

# 19. Create Submission File

In [ ]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_pred
})

submission.to_csv("submission.csv", index=False)
